In [1]:
# 加载已有的 FAISS index
import os
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.load_local(
    "../data/faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)
print("Index loaded successfully!")

/var/folders/tk/x43kyqbn33v_ttx5rckk29cm0000gn/T/ipykernel_72976/2920114748.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Index loaded successfully!


In [2]:
# 连接 Ollama + 设计 Prompt
import requests
import json

def query_mistral(prompt: str) -> str:
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "mistral",
            "prompt": prompt,
            "stream": False
        }
    )
    return response.json()["response"]

def rag_pipeline(query: str, k: int = 3) -> dict:
    # 检索
    retriever = vectorstore.as_retriever(search_kwargs={"k": k})
    docs = retriever.invoke(query)
    
    # 拼接 context
    context = "\n\n".join([doc.page_content for doc in docs])
    sources = [f"{doc.metadata.get('source','?')} p.{doc.metadata.get('page','?')}" 
               for doc in docs]
    
    # Prompt
    prompt = f"""You are a cybersecurity advisor helping practitioners.
Use ONLY the context below to answer the question.
If the context does not contain enough information, say so.

Context:
{context}

Question: {query}

Provide exactly 3 concise, actionable guidelines for cybersecurity practitioners.
Format as:
Guideline 1: ...
Guideline 2: ...
Guideline 3: ...
"""
    answer = query_mistral(prompt)
    
    return {
        "query": query,
        "answer": answer,
        "sources": sources,
        "retrieved_chunks": [doc.page_content for doc in docs]
    }

print("Pipeline ready!")

Pipeline ready!


In [3]:
# 测试 RAG pipeline
result = rag_pipeline("How should an organization respond to a phishing email?")

print("=== QUERY ===")
print(result["query"])
print("\n=== GENERATED GUIDELINES ===")
print(result["answer"])
print("\n=== SOURCES ===")
for s in result["sources"]:
    print(f"  - {s}")

=== QUERY ===
How should an organization respond to a phishing email?

=== GENERATED GUIDELINES ===
 Guideline 1: Immediately report the received phishing email to the IT department or security team. This will allow them to analyze the email and determine if other users have been affected, and take necessary actions to secure the organization's systems.

Guideline 2: Do not click on any links or download attachments within the suspected phishing email. These could potentially lead to further compromise of sensitive data or installation of malware.

Guideline 3: Educate employees about common signs of phishing attempts and remind them never to provide personal information, login credentials, or financial details in response to an email request. Regular training and awareness campaigns can help reduce the likelihood of successful phishing attacks.

=== SOURCES ===
  - ../data/nist_sp800_177r1.pdf p.30
  - ../data/nist_sp800_177r1.pdf p.34
  - ../data/nist_sp800_177r1.pdf p.40
